In [8]:
import time
from pathlib import Path

import pandas as pd
import requests

In [9]:
FILE_1 = "CommercialTransaction20260303143736.csv"
FILE_2 = "CommercialTransaction20260303143938.csv"
MERGED_COORDS = "commercial_transactions_merged_clean_with_coords.csv"
COORDS_FILE = "retail_transactions.csv"

base = Path.cwd()

df1 = pd.read_csv(base / FILE_1)
df2 = pd.read_csv(base / FILE_2)

df = pd.concat([df1, df2], ignore_index=True)
print(f"Merged rows: {len(df)}")
df.head()

Merged rows: 1620


,Project Name,Street Name,Property Type,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Type of Area,Area (SQM),Unit Price ($ PSM),Tenure,Postal District,Floor Level
0,THE BENCOOLEN,BENCOOLEN STREET,Retail,"945,000",161.46,"5,853",Feb-26,Strata,15,"63,000",99 yrs lease commencing from 1995,7,01 to 05
1,PARKLANE SHOPPING MALL,SELEGIE ROAD,Retail,"800,000",753.48,"1,062",Feb-26,Strata,70,"11,429",99 yrs lease commencing from 1974,7,06 to 10
2,LUCKY PLAZA,ORCHARD ROAD,Retail,"3,500,000",548.96,"6,376",Feb-26,Strata,51,"68,627",Freehold,9,01 to 05
3,CITY GATE,BEACH ROAD,Retail,"900,000",398.27,"2,260",Feb-26,Strata,37,"24,324",99 yrs lease commencing from 2014,7,B1 to B5
4,LUCKY PLAZA,ORCHARD ROAD,Retail,"4,208,800",495.14,"8,500",Feb-26,Strata,46,"91,496",Freehold,9,B1 to B5


## Clean fields and prepare helper functions
This cell defines numeric/date cleaning and geocoding helpers (OneMap API).

In [10]:
def to_number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.replace('"', "", regex=False).str.strip(),
        errors="coerce",
    )


def geocode_onemap(query: str, timeout: int = 10):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {"searchVal": query, "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": 1}
    try:
        r = requests.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])
        if not results:
            return None, None, None
        top = results[0]
        lat = pd.to_numeric(top.get("LATITUDE"), errors="coerce")
        lon = pd.to_numeric(top.get("LONGITUDE"), errors="coerce")
        address = top.get("ADDRESS")
        return lat, lon, address
    except Exception:
        return None, None, None


def build_query(row: pd.Series) -> str:
    project = str(row.get("Project Name", "")).strip()
    street = str(row.get("Street Name", "")).strip()
    district = str(row.get("Postal District", "")).strip()

    candidates = [
        f"{project} {street} Singapore",
        f"{street} Singapore",
        f"District {district} Singapore" if district else "",
    ]
    for c in candidates:
        if c.strip() and c.strip() != "N.A. Singapore":
            return c
    return "Singapore"

## Run merge + cleaning + coordinates enrichment

In [11]:
numeric_cols = [
    "Transacted Price ($)",
    "Area (SQFT)",
    "Unit Price ($ PSF)",
    "Area (SQM)",
    "Unit Price ($ PSM)",
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = to_number(df[c])

if "Sale Date" in df.columns:
    df["Sale Date"] = pd.to_datetime(df["Sale Date"], format="%b-%y", errors="coerce")

for c in ["Project Name", "Street Name", "Property Type", "Tenure", "Floor Level", "Postal District", "Type of Area"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

dedup_keys = [
    "Project Name",
    "Street Name",
    "Property Type",
    "Transacted Price ($)",
    "Area (SQFT)",
    "Sale Date",
    "Postal District",
    "Floor Level",
]
dedup_keys = [c for c in dedup_keys if c in df.columns]
df = df.drop_duplicates(subset=dedup_keys)

df["latitude"] = pd.NA
df["longitude"] = pd.NA
df["geocoded_address"] = pd.NA

cache = {}
for idx, row in df.iterrows():
    q = build_query(row)
    if q in cache:
        lat, lon, addr = cache[q]
    else:
        lat, lon, addr = geocode_onemap(q)
        cache[q] = (lat, lon, addr)
        time.sleep(0.25)

    df.at[idx, "latitude"] = lat
    df.at[idx, "longitude"] = lon
    df.at[idx, "geocoded_address"] = addr

df = df.sort_values(by=["Sale Date", "Project Name"], ascending=[False, True], na_position="last")
df.to_csv(base / MERGED_COORDS, index=False)

important_cols = [
    "Project Name",
    "Street Name",
    "Property Type",
    "Sale Date",
    "Transacted Price ($)",
    "Unit Price ($ PSF)",
    "Area (SQFT)",
    "Postal District",
    "Floor Level",
    "Tenure",
    "latitude",
    "longitude",
    "geocoded_address",
]
important_cols = [c for c in important_cols if c in df.columns]
df_slim = df[important_cols].copy()

price_bins = [-float("inf"), 1_000_000, 3_000_000, 10_000_000, float("inf")]
price_labels = ["<1M", "1-3M", "3-10M", ">10M"]
df_slim["price_level"] = pd.cut(
    df_slim["Transacted Price ($)"],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
    ordered=True,
).astype("category")

psf_bins = [-float("inf"), 2000, 4000, 7000, float("inf")]
psf_labels = ["<2k PSF", "2-4k PSF", "4-7k PSF", ">7k PSF"]
if "Unit Price ($ PSF)" in df_slim.columns:
    df_slim["psf_level"] = pd.cut(
        df_slim["Unit Price ($ PSF)"],
        bins=psf_bins,
        labels=psf_labels,
        include_lowest=True,
        ordered=True,
    ).astype("category")
    
df_slim = df_slim.dropna(subset=["latitude", "longitude"], how="all")
df_slim = df_slim.dropna(subset=["Unit Price ($ PSF)"], how="all")

df_slim.to_csv(base / COORDS_FILE, index=False)



print(f"Saved full: {base / MERGED_COORDS}")
print(f"Saved slim: {base / COORDS_FILE}")
print(f"Rows: {len(df_slim)}")
print(f"Missing coordinates: {df_slim['latitude'].isna().sum()}")
df_slim.head()

Saved full: /home/beary/projects/test/commercial_transactions_merged_clean_with_coords.csv
Saved slim: /home/beary/projects/test/retail_transactions.csv
Rows: 1304
Missing coordinates: 0


,Project Name,Street Name,Property Type,Sale Date,Transacted Price ($),Unit Price ($ PSF),Area (SQFT),Postal District,Floor Level,Tenure,latitude,longitude,geocoded_address,price_level,psf_level
1212,BEACH ROAD CONSERVATION AREA,PURVIS STREET,Shop House,2026-02-01,14000000,8407.0,1665.19,7,-,999 yrs lease commencing from 1827,1.29675,103.85514,7 PURVIS STREET BEACH ROAD CONSERVATION AREA S...,>10M,>7k PSF
5,BUKIT TIMAH PLAZA,JALAN ANAK BUKIT,Retail,2026-02-01,65395697,1459.0,44810.53,21,B1 to B5,99 yrs lease commencing from 1976,1.338759,103.7786,1 JALAN ANAK BUKIT BUKIT TIMAH PLAZA SINGAPORE...,>10M,<2k PSF
3,CITY GATE,BEACH ROAD,Retail,2026-02-01,900000,2260.0,398.27,7,B1 to B5,99 yrs lease commencing from 2014,1.302316,103.862332,371 BEACH ROAD CITY GATE SINGAPORE 199597,<1M,2-4k PSF
6,CORONATION SHOPPING PLAZA,BUKIT TIMAH ROAD,Retail,2026-02-01,32386440,3177.0,10193.51,10,01 to 05,Freehold,1.323835,103.81,587 BUKIT TIMAH ROAD CORONATION SHOPPING PLAZA...,>10M,2-4k PSF
10,FAR EAST SHOPPING CENTRE,ORCHARD ROAD,Retail,2026-02-01,1900000,4203.0,452.09,9,01 to 05,999 yrs lease commencing from 1871,1.305506,103.829985,545 ORCHARD ROAD FAR EAST SHOPPING CENTRE SING...,1-3M,4-7k PSF


## Visualize coordinates on an interactive map
Use Folium to plot transaction points from the merged CSV.

In [12]:
%pip install folium -q
import folium
from folium.plugins import MarkerCluster

Note: you may need to restart the kernel to use updated packages.


In [13]:
from IPython.display import IFrame, display

map_source = base / COORDS_FILE if (base / COORDS_FILE).exists() else base / MERGED_COORDS
map_df = pd.read_csv(map_source)

map_df["latitude"] = pd.to_numeric(map_df["latitude"], errors="coerce")
map_df["longitude"] = pd.to_numeric(map_df["longitude"], errors="coerce")
map_df = map_df.dropna(subset=["latitude", "longitude"]).copy()

if map_df.empty:
    raise ValueError("No valid coordinates found. Check latitude/longitude columns.")

center = [map_df["latitude"].mean(), map_df["longitude"].mean()]
m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(m)

for _, r in map_df.iterrows():
    popup = (
        f"Project: {r.get('Project Name', 'N/A')}<br>"
        f"Street: {r.get('Street Name', 'N/A')}<br>"
        f"Type: {r.get('Property Type', 'N/A')}<br>"
        f"Price: {r.get('Transacted Price ($)', 'N/A')}<br>"
        f"Price Level: {r.get('price_level', 'N/A')}<br>"
        f"PSF Level: {r.get('psf_level', 'N/A')}<br>"
        f"Sale Date: {r.get('Sale Date', 'N/A')}"
    )
    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=3,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        popup=popup,
    ).add_to(cluster)

map_output = base / "commercial_transactions_map.html"
m.save(map_output)
print(f"Map source: {map_source}")
print(f"Map saved to: {map_output}")

display(m)
IFrame(src=map_output.as_posix(), width=1000, height=650)

Map source: /home/beary/projects/test/retail_transactions.csv
Map saved to: /home/beary/projects/test/commercial_transactions_map.html


# Add to supabase

In [13]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "database" / "retail_transactions.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find database/retail_transactions.csv from current working directory or its parent folders."
    )

cwd = Path.cwd()
repo_root = find_repo_root(cwd)
csv_path = repo_root / "database" / "retail_transactions.csv"

df = pd.read_csv(csv_path)
print(f"Working directory: {cwd}")
print(f"CSV path: {csv_path}")
df.info()

Working directory: /home/beary/projects/SmartLocateSG/backend/scripts
CSV path: /home/beary/projects/SmartLocateSG/database/retail_transactions.csv
<class 'pandas.DataFrame'>
RangeIndex: 1304 entries, 0 to 1303
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Project Name          1304 non-null   str    
 1   Street Name           1304 non-null   str    
 2   Property Type         1304 non-null   str    
 3   Sale Date             1304 non-null   str    
 4   Transacted Price ($)  1304 non-null   int64  
 5   Unit Price ($ PSF)    1304 non-null   float64
 6   Area (SQFT)           1304 non-null   float64
 7   Postal District       1304 non-null   int64  
 8   Floor Level           1304 non-null   str    
 9   Tenure                1304 non-null   str    
 10  latitude              1304 non-null   float64
 11  longitude             1304 non-null   float64
 12  geocoded_address      1304 non-null  